# Day 3 – Your First Agent: LangChain

**Workshop:** One Person Company – AI Agents Workshop  
**Institution:** Aditya University – DoTT  
**Project:** Sweet Crumbs Home Bakery  
**Day:** 3  
**Tools:** Python, Google Colab, LangChain, Groq API

## Learning Objectives

Today I learned how to:

1. Understand the difference between a chatbot and an AI agent.
2. Identify the four main components of an AI agent.
3. Understand the Think → Act → Observe → Repeat agent loop.
4. Differentiate Traditional AI, Generative AI and Agentic AI.
5. Understand when an AI agent is required.
6. Learn the basics of LangChain.
7. Work with models, prompts and chains.
8. Implement conversation history and memory.
9. Create a CSV-based testing dataset.
10. Evaluate chatbot responses using automated testing.
11. Improve system prompts based on test results.
12. Experiment with LLM temperature.
13. Create Python functions with type hints and docstrings for future agent tools.

## Project Objective

Build and evaluate a LangChain-powered assistant for Sweet Crumbs Home Bakery and prepare Python functions that can be connected as tools to an AI agent in the next session.

## 1. Installing and Setting Up LangChain

### What is LangChain?

LangChain is a Python framework used to build applications powered by Large Language Models (LLMs).

It provides building blocks that help connect models, prompts, tools and memory.

### Installation

We install the following packages:

- `langchain` – Provides the main LangChain framework.
- `langchain-groq` – Connects LangChain with Groq models.

We use Google Colab and the Groq API for this practical.

In [1]:
!pip install -q langchain langchain-groq

### Connecting to Groq

We use Groq as the LLM provider.

The API key is retrieved securely from Google Colab Secrets instead of being directly written in the notebook.

The `ChatGroq` class allows LangChain to communicate with the selected Groq model.

**Model:** Use the same Groq model that worked in your practical.

**Temperature:** 0.3 for relatively consistent responses.

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("LangChain and Groq setup completed!")

LangChain and Groq setup completed!


### Testing the Language Model

Before building the chatbot, we first check whether the LLM is responding correctly.

The `invoke()` method sends a message to the model and returns its response.

We use a simple question asking the model to explain what an AI agent is.

In [3]:
reply = llm.invoke("Explain what an AI agent is in two simple sentences.")

print(reply.content)

An AI agent is a software program that can perceive its environment and take actions to achieve goals. It uses algorithms and data to make decisions and learn from experience.


## 2. Understanding Chatbots and AI Agents

### What is a Chatbot?

A chatbot is a computer program that communicates with users and provides responses to their questions.

**Example:**

Customer: What is the price of a chocolate cake?

Chatbot: A chocolate cake costs Rs 650.

### What is an AI Agent?

An AI agent is a system that can understand a goal, make decisions, use tools and perform actions to complete a task.

**Example:**

Customer: I need three chocolate cakes.

An AI agent can:
1. Understand the customer's request.
2. Check the bakery menu.
3. Calculate the total price.
4. Prepare an order summary.

### Difference Between a Chatbot and an AI Agent

| Chatbot | AI Agent |
|---|---|
| Mainly responds to questions | Works towards a goal |
| Usually follows a conversation flow | Can decide which action is needed |
| May not use external tools | Can use tools |
| Provides information | Can perform multiple steps |

**Learning:** A chatbot mainly communicates, while an AI agent can use tools and perform actions to achieve a goal.

## 3. Four Main Components of an AI Agent

An AI agent has four important components.

### 1. Brain – LLM

The Large Language Model understands user messages and helps decide what to do.

### 2. Instructions – System Prompt

Instructions define the agent's role, business information, rules and response format.

### 3. Tools – Python Functions

Tools allow the agent to perform tasks such as calculating prices and checking availability.

### 4. Memory – Conversation History

Memory helps the agent remember previous messages and maintain context.

**Sweet Crumbs Example:**

- Brain: Groq LLM
- Instructions: Sweet Crumbs system prompt
- Tools: Python functions
- Memory: Conversation history

## 4. How an AI Agent Works – Agent Loop

An AI agent follows a repeated process:

**Think → Act → Observe → Repeat**

1. Think – Understand the request and decide what to do.
2. Act – Use an available tool to perform the action.
3. Observe – Receive and understand the tool's result.
4. Repeat – Continue if more actions are required.

When the goal is achieved, the agent provides the final response.

**Example:** A customer requests three plum cakes. The agent can check the price, calculate the total and prepare an order summary.

## 5. Traditional AI, Generative AI and Agentic AI

### 1. Traditional AI

Traditional AI performs specific tasks using predefined rules.

Example: A Python program calculates the total price of five cupcakes.

### 2. Generative AI

Generative AI creates new content based on user instructions.

Example: Generating a birthday message for a bakery customer.

### 3. Agentic AI

Agentic AI can understand a goal, make decisions, use tools and perform multiple actions.

Example: Understanding a customer's order, checking prices, calculating the bill and preparing an order summary.

**Learning:** Traditional AI follows rules, Generative AI creates content, and Agentic AI can plan and act using tools.

## 6. When Do We Actually Need an AI Agent?

Not every task requires an AI agent. We should choose the simplest approach that can complete the task.

| Business Task | Suitable Approach |
|---|---|
| Calculate the price of 5 cupcakes | Plain Python Code |
| Generate a birthday cake message | One LLM Call |
| Understand an order, check the menu, calculate the bill and prepare an order summary | AI Agent |

**Learning:** Use Python for fixed calculations, one LLM call for simple content generation, and an AI agent when a task requires multiple decisions and actions.

## 7. Building a LangChain Chatbot

### Part A – Invoking the Chat Model

The `invoke()` method sends messages directly to the LLM.

We can provide different message roles:

- `system` – Instructions for the model.
- `human` – The user's question.

The model processes these messages and generates a response.

In [4]:
reply = llm.invoke([
    ("system", "You are Sweet Crumbs Home Bakery's friendly assistant. Answer in under 30 words."),
    ("human", "Do you make eggless cakes?")
])
print(reply.content)

Yes, we offer a variety of delicious egg‑free cakes, from chocolate to vanilla. Let us know your flavor preference and we’ll bake a perfect eggless treat for you!


### Part B – Creating a Prompt Template

A prompt template allows us to create reusable prompts with placeholders.

In this practical, we use:

- `{business}` – The business name.
- `{question}` – The customer's question.

`ChatPromptTemplate.from_messages()` creates a reusable message template.

This makes it easier to use the same chatbot structure for different businesses.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the assistant for {business}. Answer in under 30 words."),
    ("human", "{question}")
])

### Part C – Connecting Components Using a Chain

LangChain allows us to connect components using the pipe operator (`|`).

The chain connects three components:

1. Prompt – Prepares the messages.
2. LLM – Generates the response.
3. StrOutputParser – Converts the model output into a normal string.

**Flow:**

Prompt → LLM → Output Parser

The `chain.invoke()` method executes the complete workflow.

In [6]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "business": "Sweet Crumbs Home Bakery",
    "question": "Do you make eggless cakes?"
})

print(answer)

Yes, we offer a variety of egg‑free cakes—from chocolate to vanilla—made with plant‑based substitutes. Let us know your flavor preference!


In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

## 8. Building the Sweet Crumbs Chatbot

Now we create a system prompt containing the bakery's business information and response rules.

The prompt defines:

- ROLE – The identity of the chatbot.
- FACTS – The bakery menu and prices.
- RULES – What the chatbot must and must not do.
- FORMAT – How the chatbot should respond.

The chatbot must not invent business information or make unsupported promises.

In [8]:
SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake: Rs 500
- Chocolate cake: Rs 650
- Cupcake: Rs 80 each
- Orders need 24 hours' notice.
- All cakes have eggless versions.

RULES:
- Never promise a delivery time or place.
- If information is not available, say "I'll check with the owner."
- Work out totals carefully.
- Include prices as plain numbers without commas, such as Rs 1000.
- When explaining order notice, explicitly mention "24 hours".
- Never claim that delivery is available to any location.
  For delivery questions, always say "I'll check with the owner."

FORMAT:
- Under 50 words.
- Be friendly.
- Reply in English or Telugu, matching the customer."""

### Creating a Prompt with Conversation History

`MessagesPlaceholder("history")` creates a place in the prompt for previous conversation messages.

The prompt now contains:

1. System instructions.
2. Previous conversation history.
3. The customer's current question.

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

In [10]:
chain = prompt | llm | StrOutputParser()

### Implementing Conversation Memory

We use a Python list named `history` to store previous messages.

LangChain provides two message objects:

- `HumanMessage` – Stores the customer's message.
- `AIMessage` – Stores the assistant's response.

The `ask()` function sends the previous history along with the new question and stores the latest question and answer.

This allows the chatbot to maintain context during a conversation.

In [11]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])

    return answer

### Testing Conversation Memory

We test whether the chatbot can remember a customer's name across multiple messages.

First, the customer introduces themselves as Ravi. Then the customer asks the chatbot to recall their name.

### Observation

The chatbot successfully remembered Ravi's name using the stored conversation history.

**Learning:** Conversation history helps an LLM maintain context across multiple messages.

**Limitation:** This memory is stored in a Python list and is temporary. It will be lost when the Colab runtime is restarted.

In [12]:
print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What is my name?"))

Yes, all our cakes are eggless. Please give 24 hours notice for orders.
I’m not sure—I'll check with the owner.


## 9. Creating a Testing Dataset

A chatbot should be tested using different types of customer questions.

For this practical, we create a CSV file named `tests.csv`.

The dataset contains two columns:

- `question` – The customer's question.
- `must_include` – A keyword or value expected in the chatbot's response.

We created 10 test cases covering product prices, eggless availability, advance notice, quantity calculations, delivery questions and menu information.

This dataset helps us evaluate the chatbot automatically.

In [13]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
What is the price of a chocolate cake?,650
How much is one cupcake?,80
Do you make eggless cakes?,eggless
Can I order a cake for today evening?,24 hours
How much do 6 cupcakes cost?,480
What is the price of 2 plum cakes?,1000
Do you deliver to Kakinada?,check with the owner
What flavours of cakes do you offer?,plum
How much do a chocolate cake and 2 cupcakes cost?,810

Overwriting tests.csv


### Viewing the Testing Dataset

We use Pandas to read and display the CSV file.

This allows us to verify that all 10 questions and their expected keywords have been loaded correctly.

In [14]:
import pandas as pd

tests = pd.read_csv("tests.csv")
display(tests)

,question,must_include
0,How much is a plum cake?,500
1,What is the price of a chocolate cake?,650
2,How much is one cupcake?,80
3,Do you make eggless cakes?,eggless
4,Can I order a cake for today evening?,24 hours
5,How much do 6 cupcakes cost?,480
6,What is the price of 2 plum cakes?,1000
7,Do you deliver to Kakinada?,check with the owner
8,What flavours of cakes do you offer?,plum
9,How much do a chocolate cake and 2 cupcakes cost?,810


## 10. Creating an Automated Testing Function

We use Python's built-in `csv` module to read the testing dataset.

The `run_tests()` function:

1. Reads questions from `tests.csv`.
2. Sends each question to the chatbot.
3. Checks whether the expected keyword appears in the answer.
4. Marks each result as PASS or FAIL.
5. Calculates the overall pass percentage.

### Pass Rate Formula

Pass Rate = (Passed Tests / Total Tests) × 100

We use an empty history for each test so that one test does not influence another.

In [15]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    with open(path, "r") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        ok = expected in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "→",
            answer[:70]
        )

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [16]:
run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500. Let me know if you’d like to place an order;
PASS | What is the price of a chocolate cake? → The chocolate cake costs Rs 650.
PASS | How much is one cupcake? → One cupcake costs Rs 80.
PASS | Do you make eggless cakes? → Yes, all our cakes—including plum, chocolate, and cupcakes—are availab
PASS | Can I order a cake for today evening? → I’m sorry, we need 24 hours notice for orders. All cakes have eggless 
PASS | How much do 6 cupcakes cost? → 6 cupcakes cost Rs 480.
PASS | What is the price of 2 plum cakes? → The price for 2 plum cakes is Rs 1000.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What flavours of cakes do you offer? → We offer plum cake (Rs 500), chocolate cake (Rs 650), and cupcakes (Rs
PASS | How much do a chocolate cake and 2 cupcakes cost? → A chocolate cake (Rs 650) plus 2 cupcakes (Rs 80 each) totals Rs 810. 

10/10 passed (100%)


## 11. Prompt Engineering and Improvement

Prompt engineering is the process of improving instructions given to an LLM to obtain more reliable responses.

We tested the Sweet Crumbs chatbot using 10 customer questions.

### Problems Identified

The initial testing revealed:
1. The response to a same-day order question did not always contain the exact expected phrase.
2. Price formatting sometimes included commas, causing keyword matching failures.
3. The chatbot incorrectly claimed delivery was available to Kakinada.

### Improvements Made

We improved the system prompt by adding:
- Instructions to include prices as plain numbers without commas.
- A rule to explicitly mention "24 hours" for order notice questions.
- A rule never to claim delivery availability without confirmed information.
- A requirement to say "I'll check with the owner" for delivery questions.

### Results

| Prompt Version | Passed Tests | Pass Rate |
|---|---:|---:|
| Initial Prompt | 8/10 | 80% |
| Improved Prompt – V2 | 9/10 | 90% |
| Improved Prompt – V3 | 10/10 | 100% |

### Observation

The chatbot's test pass rate improved from 80% to 100% after refining its instructions.

**Learning:** Prompt engineering is an iterative process: test, identify problems, improve the prompt and test again.

**Note:** A keyword-based test can fail even when an answer is correct. For example, "Rs 1,000" and "Rs 1000" represent the same amount but may not match the same expected keyword.

## 12. Experimenting with Temperature

Temperature is a model setting that influences the randomness of generated responses.

- Lower temperature generally produces more consistent responses.
- Higher temperature can produce more varied and creative responses.

For this experiment, we changed the model temperature from 0.3 to 1.0 and ran the same automated tests twice.

### Experiment Results

| Test Run | Temperature | Passed Tests | Pass Rate |
|---|---:|---:|---:|
| Run 1 | 1.0 | 9/10 | 90% |
| Run 2 | 1.0 | 10/10 | 100% |

### Observation

The responses were worded differently between runs, even though the questions and prompt remained the same.

The first run had one keyword-matching failure, while the second run passed all 10 tests.

**Learning:** LLM responses can vary between runs. Repeated testing helps us understand the consistency of a chatbot.

**Note:** Higher temperature does not guarantee a lower pass rate or a different answer every time.

In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

run_tests()

PASS | How much is a plum cake? → The plum cake costs Rs 500.
PASS | What is the price of a chocolate cake? → Sure! A chocolate cake costs Rs 650. Let me know if you’d like to orde
PASS | How much is one cupcake? → Rs 80 each.
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
PASS | Can I order a cake for today evening? → I’m sorry, we need 24 hours notice for orders. I can’t take today’s or
PASS | How much do 6 cupcakes cost? → 6 cupcakes cost Rs 480.
PASS | What is the price of 2 plum cakes? → The total for 2 plum cakes is Rs 1000.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What flavours of cakes do you offer? → We offer plum cake (Rs 500) and chocolate cake (Rs 650), plus cupcakes
PASS | How much do a chocolate cake and 2 cupcakes cost? → Chocolate cake is Rs 650. Two cupcakes cost Rs 160. Total Rs 810.

10/10 passed (100%)


In [18]:
run_tests()

PASS | How much is a plum cake? → Rs 500.
PASS | What is the price of a chocolate cake? → The chocolate cake is Rs 650.
PASS | How much is one cupcake? → One cupcake costs Rs 80. Let me know if you'd like to order!
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
PASS | Can I order a cake for today evening? → I’m sorry, we need 24 hours’ notice. Please place your order at least 
PASS | How much do 6 cupcakes cost? → Six cupcakes cost Rs 480.
PASS | What is the price of 2 plum cakes? → Two plum cakes cost Rs 1000.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What flavours of cakes do you offer? → We offer three delicious options: Plum cake (Rs 500), Chocolate cake (
PASS | How much do a chocolate cake and 2 cupcakes cost? → A chocolate cake costs Rs 650. Two cupcakes cost Rs 160. Total Rs 810.

10/10 passed (100%)


## 13. Identifying Where AI Agents Are Needed

Not every business activity requires an AI agent.

We classified three Sweet Crumbs business activities based on the simplest suitable approach.

| Business Activity | Suitable Approach |
|---|---|
| Calculate the total price of 5 cupcakes | Plain Python Code |
| Generate a birthday cake message | One LLM Call |
| Understand a customer order, check the menu, calculate the bill and prepare an order summary | AI Agent |

### Observation

- Fixed calculations are better handled using Python.
- Simple content generation can be handled using one LLM call.
- Tasks involving multiple decisions and actions may require an AI agent.

**Learning:** We should use AI agents only when the task actually needs them.

## 14. Creating Python Functions for Future Agent Tools

Before building an AI agent, we need functions that it can use as tools.

For Sweet Crumbs Home Bakery, we created two Python functions.

### 1. calculate_total()

This function calculates the total price of bakery items using their quantity and individual price.

### 2. check_availability()

This function checks whether an order meets the bakery's 24-hour advance notice requirement.

### Important Python Concepts

- Type hints describe the expected input and output types.
- Docstrings explain the purpose of a function.
- Return statements provide the result of a function.

These functions will be useful when we connect Python functions to an AI agent in Day 4.

In [19]:
def calculate_total(quantity: int, price: float) -> float:
    """
    Calculate the total price of bakery items.

    Args:
        quantity: Number of items.
        price: Price of one item.

    Returns:
        Total order price.
    """
    total = quantity * price
    return total

In [20]:
bill = calculate_total(6, 80)
print("Total bill: Rs", bill)

Total bill: Rs 480


### Testing the Availability Function

We test two situations:

1. An order requested with 48 hours of advance notice.
2. An order requested with only 5 hours of advance notice.

The function should accept the first and reject the second based on the 24-hour rule.

In [21]:
def check_availability(hours_before_order: int) -> str:
    """
    Check whether an order meets the 24-hour notice requirement.

    Args:
        hours_before_order: Hours remaining before the requested order time.

    Returns:
        Order availability message.
    """
    if hours_before_order >= 24:
        return "Order can be accepted."

    return "Order cannot be accepted. Please provide 24 hours' notice."

In [22]:
print(check_availability(48))
print(check_availability(5))

Order can be accepted.
Order cannot be accepted. Please provide 24 hours' notice.


## 15. Final Observations and Conclusion

### What I Learned Today

1. Understood the difference between a chatbot and an AI agent.
2. Learned the four main components of an AI agent: Brain, Instructions, Tools and Memory.
3. Understood the Think → Act → Observe → Repeat agent loop.
4. Learned the differences between Traditional AI, Generative AI and Agentic AI.
5. Understood when to use plain Python, one LLM call or an AI agent.
6. Learned LangChain basics, including models, prompts and chains.
7. Built a LangChain-powered chatbot using Groq.
8. Implemented conversation history using LangChain message objects.
9. Created a CSV testing dataset with 10 customer questions.
10. Improved the chatbot's test pass rate from 80% to 100% through prompt engineering.
11. Experimented with temperature 1.0 and observed variations in responses.
12. Created two Python functions with type hints and docstrings.

### Final Project Status

The Sweet Crumbs Home Bakery assistant can:

- Answer questions about bakery menu items and prices.
- Explain eggless availability.
- Explain the 24-hour advance notice requirement.
- Maintain conversation history during a session.
- Be evaluated using automated tests.

The two Python functions can calculate item totals and check the 24-hour notice rule.

### Limitations

- The chatbot does not have a real order database.
- Conversation memory is temporary and stored in a Python list.
- Automated testing uses keyword matching rather than complete semantic evaluation.
- The Python functions are not yet connected to an AI agent.

### Conclusion

Day 3 helped me understand how to build and evaluate LLM-powered applications using LangChain. I also prepared Python functions that can be connected as tools in the next stage.

**Next Step:** Build a tool-using AI agent for Sweet Crumbs Home Bakery.